In [1]:
import pandas as pd

# -----------------------------
# 1) Conjunto de dados
# -----------------------------
data = [
    ["young", "myope",        "no",  "reduced", "none"],
    ["young", "myope",        "no",  "normal",  "soft"],
    ["young", "myope",        "yes", "reduced", "none"],
    ["young", "myope",        "yes", "normal",  "hard"],
    ["young", "hypermetrope", "no",  "reduced", "none"],
    ["young", "hypermetrope", "no",  "normal",  "soft"],
    ["young", "hypermetrope", "yes", "reduced", "none"],
    ["young", "hypermetrope", "yes", "normal",  "hard"],

    ["pre-presbyopic", "myope",        "no",  "reduced", "none"],
    ["pre-presbyopic", "myope",        "no",  "normal",  "soft"],
    ["pre-presbyopic", "myope",        "yes", "reduced", "none"],
    ["pre-presbyopic", "myope",        "yes", "normal",  "hard"],
    ["pre-presbyopic", "hypermetrope", "no",  "reduced", "none"],
    ["pre-presbyopic", "hypermetrope", "no",  "normal",  "soft"],
    ["pre-presbyopic", "hypermetrope", "yes", "reduced", "none"],
    ["pre-presbyopic", "hypermetrope", "yes", "normal",  "none"],

    ["presbyopic", "myope",        "no",  "reduced", "none"],
    ["presbyopic", "myope",        "no",  "normal",  "none"],
    ["presbyopic", "myope",        "yes", "reduced", "none"],
    ["presbyopic", "myope",        "yes", "normal",  "hard"],
    ["presbyopic", "hypermetrope", "no",  "reduced", "none"],
    ["presbyopic", "hypermetrope", "no",  "normal",  "soft"],
    ["presbyopic", "hypermetrope", "yes", "reduced", "none"],
    ["presbyopic", "hypermetrope", "yes", "normal",  "none"],
]

columns = [
    "Age",
    "Spectacle prescription",
    "Astigmatism",
    "Tear production rate",
    "Recommended lenses"
]

df = pd.DataFrame(data, columns=columns)

target_col = "Recommended lenses"


# -----------------------------
# 2) Funções auxiliares do PRISM
# -----------------------------
def apply_conditions(df, conditions):
    subset = df.copy()
    for attr, value in conditions:
        subset = subset[subset[attr] == value]
    return subset


def candidate_stats(df_current, conditions, target_col, target_class, used_attributes):
    """
    Retorna estatísticas para todos os pares atributo=valor ainda disponíveis.
    Métrica principal do PRISM: p(classe_alvo | antecedente + atributo=valor)
    Desempate: maior cobertura positiva
    """
    current_subset = apply_conditions(df_current, conditions)
    candidates = []

    for attr in df_current.columns:
        if attr == target_col or attr in used_attributes:
            continue

        for value in sorted(current_subset[attr].unique()):
            candidate_subset = current_subset[current_subset[attr] == value]
            total = len(candidate_subset)
            positives = len(candidate_subset[candidate_subset[target_col] == target_class])

            if total > 0:
                prob = positives / total
                candidates.append({
                    "attribute": attr,
                    "value": value,
                    "probability": prob,
                    "positives": positives,
                    "total": total
                })

    return candidates


def choose_best_candidate(candidates):
    """
    Ordenação:
    1) maior p(classe | condição)
    2) maior cobertura positiva
    3) menor cobertura total
    4) ordem alfabética para estabilidade
    """
    return sorted(
        candidates,
        key=lambda x: (-x["probability"], -x["positives"], x["total"], x["attribute"], str(x["value"]))
    )[0]


def build_one_rule(df_remaining, target_col, target_class, verbose=True):
    """
    Constrói uma regra para uma classe alvo.
    """
    conditions = []
    used_attributes = set()

    while True:
        current_subset = apply_conditions(df_remaining, conditions)

        # Se o subconjunto ficou puro para a classe alvo, para.
        if len(current_subset) > 0 and all(current_subset[target_col] == target_class):
            break

        candidates = candidate_stats(
            df_remaining,
            conditions,
            target_col,
            target_class,
            used_attributes
        )

        if not candidates:
            break

        best = choose_best_candidate(candidates)
        conditions.append((best["attribute"], best["value"]))
        used_attributes.add(best["attribute"])

        if verbose:
            antecedente = " AND ".join([f"{a} = {v}" for a, v in conditions[:-1]]) or "(vazio)"
            print(f"\nAntecedente atual: {antecedente}")
            print(
                f"Escolha: {best['attribute']} = {best['value']} "
                f"-> p({target_class}) = {best['positives']}/{best['total']} = {best['probability']:.2f}"
            )

        current_subset = apply_conditions(df_remaining, conditions)

        # Regra perfeita
        if len(current_subset) > 0 and all(current_subset[target_col] == target_class):
            break

        # Se não há mais atributos, para
        if len(used_attributes) == len(df_remaining.columns) - 1:
            break

    covered = apply_conditions(df_remaining, conditions)
    positive_covered = covered[covered[target_col] == target_class]

    return conditions, positive_covered.index.tolist(), covered


def prism(df, target_col, class_order=None, verbose=True):
    """
    Gera regras PRISM para cada classe.
    Remove apenas exemplos positivos cobertos por cada regra.
    """
    if class_order is None:
        class_order = list(df[target_col].unique())

    all_rules = []

    for target_class in class_order:
        if verbose:
            print("\n" + "=" * 70)
            print(f"Classe alvo: {target_class}")
            print("=" * 70)

        df_remaining = df.copy()
        class_rules = []

        while True:
            positives_left = df_remaining[df_remaining[target_col] == target_class]
            if positives_left.empty:
                break

            conditions, positive_indexes, covered_subset = build_one_rule(
                df_remaining, target_col, target_class, verbose=verbose
            )

            if not conditions or len(positive_indexes) == 0:
                break

            rule = {
                "class": target_class,
                "conditions": conditions,
                "covered_positive_indexes": positive_indexes
            }
            class_rules.append(rule)
            all_rules.append(rule)

            if verbose:
                cond_text = " AND ".join([f"{a} = {v}" for a, v in conditions])
                print(f"REGRA: IF {cond_text} THEN {target_col} = {target_class}")
                print(f"Cobertura positiva: {len(positive_indexes)} exemplo(s)")

            # Remove SOMENTE os exemplos positivos cobertos por essa regra
            df_remaining = df_remaining.drop(index=positive_indexes)

        if verbose and class_rules:
            print("\nResumo das regras:")
            for i, rule in enumerate(class_rules, start=1):
                cond_text = " AND ".join([f"{a} = {v}" for a, v in rule["conditions"]])
                print(f"R{i}: IF {cond_text} THEN {target_col} = {target_class}")

    return all_rules


def predict_prism(rules, instance):
    """
    Aplica a primeira regra satisfeita.
    """
    for rule in rules:
        ok = True
        for attr, value in rule["conditions"]:
            if instance[attr] != value:
                ok = False
                break
        if ok:
            return rule["class"]
    return None


def evaluate_prism(rules, df, target_col):
    preds = df.apply(lambda row: predict_prism(rules, row), axis=1)
    acc = (preds == df[target_col]).mean()
    return acc, preds


def print_rules(rules, target_col):
    print("\n" + "=" * 70)
    print("CONJUNTO FINAL DE REGRAS")
    print("=" * 70)
    for i, rule in enumerate(rules, start=1):
        cond_text = " AND ".join([f"{a} = {v}" for a, v in rule["conditions"]])
        print(f"R{i}: IF {cond_text} THEN {target_col} = {rule['class']}")


# -----------------------------
# 3) Executar PRISM
# -----------------------------
# Ordem das classes igual à solução do PDF
rules = prism(
    df,
    target_col=target_col,
    class_order=["hard", "soft", "none"],
    verbose=True
)

print_rules(rules, target_col)

accuracy, predictions = evaluate_prism(rules, df, target_col)
print("\nAcurácia no próprio conjunto:", round(accuracy, 2))

df_result = df.copy()
df_result["Predicted"] = predictions
display(df_result)


Classe alvo: hard

Antecedente atual: (vazio)
Escolha: Astigmatism = yes -> p(hard) = 4/12 = 0.33

Antecedente atual: Astigmatism = yes
Escolha: Tear production rate = normal -> p(hard) = 4/6 = 0.67

Antecedente atual: Astigmatism = yes AND Tear production rate = normal
Escolha: Spectacle prescription = myope -> p(hard) = 3/3 = 1.00
REGRA: IF Astigmatism = yes AND Tear production rate = normal AND Spectacle prescription = myope THEN Recommended lenses = hard
Cobertura positiva: 3 exemplo(s)

Antecedente atual: (vazio)
Escolha: Age = young -> p(hard) = 1/7 = 0.14

Antecedente atual: Age = young
Escolha: Astigmatism = yes -> p(hard) = 1/3 = 0.33

Antecedente atual: Age = young AND Astigmatism = yes
Escolha: Tear production rate = normal -> p(hard) = 1/1 = 1.00
REGRA: IF Age = young AND Astigmatism = yes AND Tear production rate = normal THEN Recommended lenses = hard
Cobertura positiva: 1 exemplo(s)

Resumo das regras:
R1: IF Astigmatism = yes AND Tear production rate = normal AND Spect

,Age,Spectacle prescription,Astigmatism,Tear production rate,Recommended lenses,Predicted
0,young,myope,no,reduced,none,none
1,young,myope,no,normal,soft,soft
2,young,myope,yes,reduced,none,none
3,young,myope,yes,normal,hard,hard
4,young,hypermetrope,no,reduced,none,none
5,young,hypermetrope,no,normal,soft,soft
6,young,hypermetrope,yes,reduced,none,none
7,young,hypermetrope,yes,normal,hard,hard
8,pre-presbyopic,myope,no,reduced,none,none
9,pre-presbyopic,myope,no,normal,soft,soft
